# Week 10 Lab: Maximum Likelihood Estimation

## Learning Objectives

By the end of this lab, you will be able to:
1. Write down the log-likelihood and score function for the exponential distribution
2. Implement a log-likelihood function in Julia
3. Use **closures** to create functions of a single parameter from multi-argument functions
4. Find the MLE numerically by minimizing the negative log-likelihood (`Optim` package)
5. Find the MLE by solving the score equation (`Roots` package)

## The Exponential Distribution

We will work with data that is exponentially distributed. Recall from undergraduate statistics that the pdf of the exponential distribution is

$$
f(y; \mu) = 
\begin{cases}
\frac{1}{\mu} e^{-(y / \mu)} & \text{if } y \geq 0 \\
0 & \text{if } y < 0
\end{cases}
$$

A random variable with this pdf is called **exponentially distributed**, written $Y \sim \text{Exp}(\mu)$.

Two helpful facts about this distribution are that $E(Y) = \mu$ and $\text{Var}(Y) = \mu^2$.

## Loading packages for this notebook

We will be needing the following packages:

In [ ]:
using Distributions, Random, Plots, Optim, Roots

## Exercise 1

Write a function `random_sample` that has as arguments the sample size `n` and the parameter $\mu$ of the exponential distribution and that returns a univariate random sample.

Hint: Look at the `Distributions` package for creating an exponential distribution object, and use `rand` to draw from it. You may want to include a `seed` keyword argument for reproducibility (see Week 3).

In [ ]:
"""
    random_sample(n, μ; seed=42)

Generate a random sample of size `n` from an Exponential(`μ`) distribution.

# Arguments
- `n::Integer`: the sample size.
- `μ::Real`: the mean parameter of the exponential distribution.
- `seed::Integer`: random seed for reproducibility (default `42`).

# Returns
- `Vector{Float64}`: a vector of `n` independent draws from Exp(`μ`).
"""
function random_sample(n, μ; seed=42)
    # Hint: Random.seed!(seed), then create an Exponential distribution and draw from it

    error("Not yet implemented")
end

## Exercise 2

Create a random sample of size `5,000` using $\mu = 8$. Plot the histogram and overlay the population pdf.

In [ ]:
my_sample = nothing  # draw a sample of size 5000 with μ = 8

# YOUR CODE HERE
# Plot a normalized histogram and overlay the population pdf
# Hint: look into the normalize keyword for histogram, and use plot! to add a curve

## Exercise 3

For the exponential distribution, we have the following results:

**pdf** (for $y \geq 0$):

$$f(y|\mu) = \frac{1}{\mu} \exp\left(-\frac{y}{\mu}\right)$$

**ln of pdf:**

$$\ln f(y|\mu) = -\ln \mu - \frac{y}{\mu}$$

**log-likelihood** (for a sample $y_1, \ldots, y_N$):

$$L(\mu) = \sum_{i=1}^{N} \ln f(y_i|\mu) = -N \ln \mu - \frac{1}{\mu} \sum_{i=1}^{N} y_i = -N\!\left(\ln \mu + \frac{\bar{y}}{\mu}\right)$$

**score** (derivative of $L$ with respect to $\mu$):

$$S(\mu) = \frac{dL}{d\mu} = -\frac{N}{\mu} + \frac{1}{\mu^2} \sum_{i=1}^{N} y_i = \frac{N}{\mu^2}\left(\bar{y} - \mu\right)$$

Setting $S(\mu) = 0$ gives the MLE in closed form: $\hat{\mu} = \bar{y}$.

---

Using the log-likelihood formula above, write a function `log_likelihood` that has arguments

* `y`: a random sample from the exponential distribution;

* `μ`: the parameter of the exponential distribution;

and that returns the log-likelihood of the random sample for the parameter $\mu$.

In [ ]:
"""
    log_likelihood(y, μ)

Compute the log-likelihood of the exponential distribution for a given sample.

Uses the formula L(μ) = -N ln(μ) - (1/μ) Σyᵢ.

# Arguments
- `y::Vector{<:Real}`: a random sample from the exponential distribution.
- `μ::Real`: the mean parameter of the exponential distribution (must be positive).

# Returns
- `Float64`: the log-likelihood value.
"""
function log_likelihood(y, μ)
    # Hint: n = length(y), then translate the log-likelihood formula above

    error("Not yet implemented")
end

## Exercise 4

Write a version of `log_likelihood` that is a function in the parameter $\mu$ only. Ultimately, we will feed that version into a solver that finds the value of $\mu$ that maximizes the log-likelihood.

The best vehicle to achieve this is a **closure**. How do you code this?

There is another complication:

The solver we are going to use below only **minimizes** an objective function. It cannot maximize. Luckily, maximizing $L(\mu)$ is equivalent to minimizing $-L(\mu)$. So when you set up your closure, let it return the negative log-likelihood. Call the closure `neg_log_likelihood_closure`.

Hint: A closure is a function that returns another function. Here is the pattern:
```julia
my_closure(y) = μ -> some_expression_involving(y, μ)
```

In [ ]:
"""
    neg_log_likelihood_closure(y)

Return a closure that computes the negative log-likelihood as a function of `μ` only.

The returned function `μ -> -log_likelihood(y, μ)` is suitable for passing to a
numerical minimizer such as `Optim.optimize`.

# Arguments
- `y::Vector{<:Real}`: the observed sample (captured by the closure).

# Returns
- `Function`: a univariate function `μ -> -L(μ)`.
"""
neg_log_likelihood_closure(y) = nothing  # replace nothing with a closure

# YOUR CODE HERE

## Exercise 5

Initialize a negative log-likelihood function by running `neg_log_likelihood_closure` with a particular random sample. Call the resulting function `neg_ll`. That function is now truly a function in $\mu$ only.

Plot the negative log-likelihood for values of $\mu$ between 4 and 15. Can you spot the minimizer?

In [ ]:
neg_ll = nothing  # initialize the closure with your sample

# YOUR CODE HERE
# Plot neg_ll over the interval [4, 15]
# Can you visually identify where the minimum is?

## Exercise 6

Find the MLE by minimizing the negative log-likelihood. Use `optimize` from the `Optim` package.

Get some guidance here:
https://julianlsolvers.github.io/Optim.jl/stable/#user/minimization/#minimizing-a-univariate-function-on-a-bounded-interval

Hint: For a univariate function on a bounded interval, the syntax is
```julia
result = optimize(f, lower, upper, Brent())
```
Extract the minimizer with `Optim.minimizer(result)`.

In [ ]:
result = nothing  # use optimize to minimize neg_ll
μ_hat = nothing   # extract the minimizer

# YOUR CODE HERE
# Compare μ_hat to mean(my_sample) — they should be (nearly) equal

## Exercise 7

Instead of maximizing the log-likelihood function, you could find the root of the score function.

1. Implement a `score(y, μ)` function using the score formula from Exercise 3
2. Write a `score_closure(y)` that returns the score as a function of $\mu$ only
3. Initialize the score for your sample and plot it
4. Use `find_zero` from the `Roots` package to find the MLE

Get some guidance on `find_zero` here: https://github.com/JuliaMath/Roots.jl

In [ ]:
"""
    score(y, μ)

Compute the score (derivative of the log-likelihood) for the exponential distribution.

Uses the formula S(μ) = -N/μ + (1/μ²) Σyᵢ = (N/μ²)(ȳ - μ).

# Arguments
- `y::Vector{<:Real}`: a random sample from the exponential distribution.
- `μ::Real`: the mean parameter of the exponential distribution (must be positive).

# Returns
- `Float64`: the score evaluated at `μ`.
"""
function score(y, μ)
    # Hint: use the score formula from Exercise 3

    error("Not yet implemented")
end

"""
    score_closure(y)

Return a closure that computes the score as a function of `μ` only.

The returned function `μ -> score(y, μ)` is suitable for root-finding with `Roots.find_zero`.

# Arguments
- `y::Vector{<:Real}`: the observed sample (captured by the closure).

# Returns
- `Function`: a univariate function `μ -> S(μ)`.
"""
score_closure(y) = nothing  # replace nothing with a closure

# YOUR CODE HERE

In [ ]:
sc = nothing  # initialize the score closure with your sample

# YOUR CODE HERE
# Plot the score function over [4, 15]
# Adding a horizontal reference line at zero can help you spot the root

In [ ]:
μ_hat2 = nothing  # use find_zero to find the root of the score

# YOUR CODE HERE
# Hint: pass the score function and a bracketing interval to find_zero
# Compare μ_hat2 to mean(my_sample)